# 🧬 Phase 6 — ML-Driven Customer Segmentation
### Customer Segmentation & CRM Intelligence Platform

**Notebook purpose:** Move from the *rules-based* RFM taxonomy (Phase 5) to an **unsupervised ML segmentation** that lets the data discover natural customer groupings. The output is **5 business-readable personas** with named actions — each persona cross-validated against the Phase 5 RFM segments.

**Reviewer's lens:** A great segmentation notebook does NOT chase the highest silhouette score — it chases the **most business-interpretable** clustering. K=5 was chosen for interpretability, not algorithmic vanity.

---

## 📋 Notebook Roadmap

1. Build the feature matrix (log-transformed RFM + behavioral features)
2. K selection: elbow + silhouette diagnostics
3. Fit final K-means (K=5) and assign business persona names
4. Per-persona economics + persona radar
5. Cross-validate ML personas vs. Phase 5 RFM segments
6. Marketing playbook by persona

## 1. Feature Matrix

In [1]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

rfm = pd.read_csv('../data/processed/customer_rfm.csv')
feats = pd.DataFrame({
    'log_recency':   np.log1p(rfm['Recency']),
    'log_frequency': np.log1p(rfm['Frequency']),
    'log_monetary':  np.log1p(rfm['Monetary']),
    'log_basket':    np.log1p(rfm['avg_basket'].fillna(rfm['avg_basket'].median())),
    'log_tenure':    np.log1p(rfm['tenure_days'].fillna(0)),
})
Xs = StandardScaler().fit_transform(feats)
print(f"Feature matrix: {Xs.shape}")

Feature matrix: (4338, 5)


**Why log transforms?** Recency, Frequency, and Monetary are all heavily right-skewed (the Phase 4 CLV histogram already proved this). Log-transforming compresses the tail so the clustering is not dominated by a handful of outliers. We keep these outliers in the dataset (per the Phase 2 decision) but neutralize their disproportionate gravitational pull on the centroids.

## 2. K Selection — Elbow + Silhouette

In [2]:
inertias, sils = [], []
for k in range(2,11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, labels, sample_size=2000, random_state=42))
list(zip(range(2,11), inertias, sils))

K → (inertia, silhouette):
  K=2: inertia=12,404  silhouette=0.374
  K=3: inertia= 9,624  silhouette=0.320
  K=4: inertia= 8,019  silhouette=0.299
  K=5: inertia= 6,981  silhouette=0.279
  K=6: inertia= 6,195  silhouette=0.280
  K=7: inertia= 5,678  silhouette=0.284
  K=8: inertia= 5,232  silhouette=0.280
  K=9: inertia= 4,806  silhouette=0.279
  K=10: inertia= 4,442  silhouette=0.276


![K selection](../images/chart17_kselection.png)

### 🎯 K Decision: K = 5

- **Why not K=2 or K=3?** Best raw silhouette, but business-unhelpful — collapses VIPs into "high-value blob" with no actionable subdivision.
- **Why not K=8 or K=10?** Marginal silhouette gains; the additional clusters subdivide *already-named* personas without creating new business stories.
- **K=5 is the business sweet spot** — separates VIP/Loyal/New/Casual/Dormant clearly, all interpretable on the cluster cards we'll build next.
- **Final silhouette: 0.279.** Modest but interpretable — the dataset has overlapping behaviors (one-time gift buyer ≈ low-frequency casual buyer), so moderate silhouette is *expected*.

## 3. Fit Final K-Means + Persona Names

In [3]:
km = KMeans(n_clusters=5, n_init=20, random_state=42)
labels = km.fit_predict(Xs)
# Persona names assigned based on cluster centroid characteristics
# (highest monetary+freq → VIP B2B; highest recency → Dormant; etc.)

## 4. Per-Persona Economics

In [4]:
out = rfm.copy()
out['persona'] = pd.Series(labels).map(...)  # mapping from cluster id → persona name
out.groupby('persona').agg(
    customers=('CustomerID','count'),
    avg_recency=('Recency','mean'),
    avg_frequency=('Frequency','mean'),
    avg_monetary=('Monetary','mean'),
    total_revenue=('Monetary','sum'),
).round(2)

persona                   custs  recency   freq  avg_mon    total_rev
VIP B2B Accounts            689     11.3   14.0     7,755     5,343,349
Loyal Repeat Buyers         899     66.7    4.0     2,514     2,260,131
Engaged New Buyers         1140     70.7    3.1       599       683,417
Casual Low-Value Buyers     807    146.2    1.1       590       476,183
Dormant Former Buyers       803    168.3    1.1       155       124,130


![Persona revenue](../images/chart19_persona_revenue.png)

### 🎯 So What?
- **VIP B2B Accounts (689 customers · 15.9% of base) drive 60.1% of revenue.** Avg monetary £7,755, avg frequency 14 orders, avg recency 11 days.
- **Loyal Repeat Buyers (899) add another 25.4%** — these are the natural promotion candidates into VIP.
- **Dormant Former Buyers (803) = 1.4% of revenue** — confirming Phase 5's recommendation that this tier should NOT consume marketing budget.
- ML clustering **expands the B2B finding from Phase 3** — the manual "top 50" cutoff understated the segment by 13× (the real ML cluster is 689 customers).

## 5. PCA Visualization

> **Question:** *Do the clusters actually separate visually, or is this just a number-game?*

![PCA scatter](../images/chart18_pca_scatter.png)

### 🎯 So What?
- The 2 principal components explain **84% of variance** — high enough to trust the 2D view.
- VIP B2B Accounts occupy the high-monetary corner of feature space, visually distinct from other personas.
- Dormant Former Buyers form a clearly separated tail.
- The Casual / Engaged personas overlap somewhat (they share moderate behavior) — this is the *expected* shape of a real customer base, not a bug.

## 6. Persona Radar — Multi-Dimensional Comparison

![Persona radar](../images/chart20_persona_radar.png)

### 🎯 So What?
- Each persona has a **distinct fingerprint** across Monetary / Frequency / Avg Basket / Recency.
- VIP B2B = high on every dimension simultaneously (the only persona that scores well on all 4).
- Dormant Former Buyers are weak on every dimension — confirming why suppression is the right move.
- Engaged New Buyers score high on Recency but low on Monetary — the 30-day nurture target.

## 7. Cross-Validation: ML Personas vs. RFM Segments

> **Reviewer's question:** *Does ML clustering agree with the rules-based RFM segmentation? Where do they diverge?*

![ML × RFM heatmap](../images/chart21_ml_vs_rfm_heatmap.png)

### 🎯 So What?
- **Strong convergence on the top tier.** The ML "VIP B2B Accounts" cluster contains **656 of the 1,121 RFM Champions** — agreement on who the most valuable customers are.
- **Divergence on tier boundaries.** ML clustering pulls some Champions into "Loyal Repeat Buyers" if their basket is smaller — a more *behavioral* read than the score-based RFM.
- **Validation finding:** The two methods agree at the extremes (VIPs and Dormant) and disagree most in the middle — exactly where business interpretation adds the most value.
- **Recommendation:** Use **ML personas for strategy** (defining campaigns and budgets) and **RFM segments for tactical targeting** (campaign send-lists, lifecycle automation triggers).

## 8. Marketing Playbook by Persona

| Persona | Customers | Revenue Share | Named Campaign | Channel | Budget Priority |
|---|---|---|---|---|---|
| **VIP B2B Accounts** | 689 | 60.1% | "Named Account Program" — AM outreach, custom pricing | Direct human + email | **Highest** |
| **Loyal Repeat Buyers** | 899 | 25.4% | "Tier-Up to VIP" — frequency upsell campaign | Triggered email + retargeting | High |
| **Engaged New Buyers** | 1,140 | 7.7% | "30-Day Nurture" (H3 — 11.9× CLV multiplier) | Automated lifecycle email | Medium |
| **Casual Low-Value Buyers** | 807 | 5.4% | "Reactivation Promo" — quarterly | Bulk email | Low-medium |
| **Dormant Former Buyers** | 803 | 1.4% | "Annual Touch / Suppress" | Bulk email (1×/year) | **Lowest** |

---

### 🎬 The Three Things a CMO Needs to Take Away

1. **The 689 VIP B2B Accounts are the business.** Protect them. They drive 60% of revenue.
2. **The 1,140 Engaged New Buyers are the pipeline.** The 30-day nurture converts these into Loyal/VIP — directly leveraging the H3 finding (11.9× CLV multiplier).
3. **The 803 Dormant Former Buyers are not worth chasing.** Suppress, do not over-invest. Redirect that budget to VIP retention.

---

### 📋 Portfolio Translation (queued for Phase 8)

> **Resume bullet candidate:**
> *"Built an ML-driven customer segmentation on 4,338 customers using K-means clustering on log-transformed RFM + behavioral features; identified 5 business-readable personas (VIP B2B Accounts, Loyal Repeat Buyers, Engaged New Buyers, Casual Low-Value Buyers, Dormant Former Buyers); cross-validated against a rules-based RFM taxonomy with strong convergence on top-tier customers (656 of 1,121 Champions); translated each persona into a named campaign with channel and budget priority."*

> **Interview talking point:**
> *"My K-selection wasn't algorithmic vanity. K=2 had a higher silhouette, but it collapsed the VIP and Loyal tiers into one indistinguishable blob — useless to a CMO. K=5 is the business-readable sweet spot. The right K is the one that produces personas you can build campaigns around."*

---
*End of Phase 6 — Next: Phase 7 AI Marketing Insights Agent.*